# HRS Silver CDM DDL SQL Generation Functional Specification
**"What information do I provide to generate DDL?"**
---

## Table of Contents

1. [Document Information](#1-document-information)
2. [Objective](#2-objective)
3. [Scope](#3-scope)
   * [3.1 Included](#31-included)
   * [3.2 Excluded](#32-excluded)
4. [SQL Generation Principles](#4-sql-generation-principles)
5. [Source Table Parameters](#5-source-table-parameters)
   * [5.1 Source Table Parameters](#51-source-table-parameters)
6. [Target Table Parameters](#6-target-table-parameters)
   * [6.1 User-Supplied Parameters](#61-user-supplied-parameters)
   * [6.2 Target Table Parameters](#62-target-table-parameters)
   * [6.3 Parameter Usage](#63-parameter-usage)
   * [6.4 Fully Qualified Target Table](#64-fully-qualified-target-table)
7. [Parent Table Dependencies](#7-parent-table-dependencies)
   * [7.1 Parent Tables](#71-parent-tables)
   * [7.2 Parent Table Keys](#72-parent-table-keys)
   * [7.3 Parent-Child Relationships](#73-parent-child-relationships)
   * [7.4 Target Foreign Keys](#74-target-foreign-keys)
   * [7.5 Natural Identifier Relationships](#75-natural-identifier-relationships)
   * [7.6 Business Grain](#76-business-grain)
8. [Target Column Definitions](#8-target-column-definitions)
   * [8.1 Column Definition Rules](#81-column-definition-rules)
   * [8.2 Identity Primary Key](#82-identity-primary-key)
   * [8.3 Foreign Key Columns](#83-foreign-key-columns)
   * [8.4 Audit Columns](#84-audit-columns)
   * [8.5 Natural Identifier Columns](#85-natural-identifier-columns)
   * [8.6 Business Columns](#86-business-columns)
9. [Source-to-Target Metadata](#9-source-to-target-metadata)
   * [9.1 Wave-Specific Variables](#91-wave-specific-variables)
   * [9.2 Wave-Invariant Variables](#92-wave-invariant-variables)
   * [9.3 Source Mapping Metadata](#93-source-mapping-metadata)
10. [Constraint Requirements](#10-constraint-requirements)
    * [10.1 Constraint Categories](#101-constraint-categories)
    * [10.2 Primary Key](#102-primary-key)
    * [10.3 Foreign Keys](#103-foreign-keys)
    * [10.4 Business Key](#104-business-key)
11. [Column Constraint Rules](#11-column-constraint-rules)
    * [11.1 Identity Column](#111-identity-column)
    * [11.2 Foreign Key Columns](#112-foreign-key-columns)
    * [11.3 Audit Columns](#113-audit-columns)
    * [11.4 Natural Identifier Columns](#114-natural-identifier-columns)
    * [11.5 Business Columns](#115-business-columns)
12. [Table and Column Comments](#12-table-and-column-comments)
    * [12.1 Table Comment](#121-table-comment)
    * [12.2 Column Comments](#122-column-comments)
13. [DDL Generation Requirements](#13-ddl-generation-requirements)
    * [13.1 Statement Order](#131-statement-order)
    * [13.2 DROP TABLE Requirement](#132-drop-table-requirement)
    * [13.3 CREATE TABLE Requirement](#133-create-table-requirement)
14. [Unsupported DDL Objects](#14-unsupported-ddl-objects)
15. [SQL Formatting Requirements](#15-sql-formatting-requirements)
    * [15.1 Keywords](#151-keywords)
    * [15.2 Indentation](#152-indentation)
    * [15.3 Column Definitions](#153-column-definitions)
    * [15.4 Constraint Definitions](#154-constraint-definitions)
    * [15.5 Object Qualification](#155-object-qualification)
    * [15.6 SQL Termination](#156-sql-termination)
16. [DDL Validation Requirements](#16-ddl-validation-requirements)
    * [16.1 Table Validation](#161-table-validation)
    * [16.2 Column Validation](#162-column-validation)
    * [16.3 Constraint Validation](#163-constraint-validation)
    * [16.4 Parent Table Validation](#164-parent-table-validation)
    * [16.5 Validation Scope](#165-validation-scope)
17. [DDL Deliverables](#17-ddl-deliverables)
18. [Generated DDL Structure](#18-generated-ddl-structure)
19. [Generation Rules Summary](#19-generation-rules-summary)
20. [Future DML Specification Boundary](#20-future-dml-specification-boundary)
21. [Final Generation Instruction](#21-final-generation-instruction)

---

## 1. Document Information

| Property           | Value                                                      |
| ------------------ | ---------------------------------------------------------- |
| Document Name      | HRS Silver CDM DDL SQL Generation Functional Specification |
| Version            | 1.0                                                        |
| Author             | Perez                                                      |
| Last Updated       | 2026-09-07                                                 |
| Target Platform    | Databricks                                                 |
| Target Runtime     | Runtime version client.5.12 (future: Databricks Runtime 15.x) |
| SQL Dialect        | Databricks SQL / Spark SQL                                 |
| Storage Format     | Delta Lake                                                 |
| Specification Type | DDL Only                                                   |

---

# 2. Objective: 

The purpose of this specification is to define the requirements for generating Databricks SQL DDL for a Silver Common Data Model (CDM) table derived from the RAND HRS dataset.

The generated DDL must create a fully defined Delta managed table containing:

* A system-generated surrogate primary key
* Parent-table foreign keys
* Natural identifiers where required
* Business attributes
* Audit columns
* Appropriate data types
* Nullability definitions
* Primary-key and foreign-key relationships
* Table and column comments
* Required table properties and constraints

The generated SQL must be executable in the specified Databricks environment.

---

# 3. Scope

## 3.1 Included

This specification defines requirements for:

1. Target table creation
2. Target table naming
3. Catalog and schema placement
4. Managed Delta table definition
5. Column definitions
6. Data types
7. Nullability
8. Identity column generation
9. Primary key definition
10. Foreign key definitions
11. Business-key definition
12. Audit columns
13. Natural identifier columns
14. Business columns
15. Table comments
16. Column comments
17. SQL formatting
18. DDL validation
19. DDL output-file requirements

## 3.2 Excluded

This specification does **not** define data-loading or transformation logic.

The following are outside the scope of this DDL specification:

* `INSERT`
* `UPDATE`
* `DELETE`
* `MERGE`
* Source-to-target transformation logic
* Source-to-parent lookup logic
* Unpivot operations
* Data cleansing
* NULL transformation rules
* Duplicate-record handling during loading
* Incremental loading
* Data-quality processing
* ETL orchestration
* Data-loading validation

These requirements will be defined in a separate DML specification.

---

# 4. SQL Generation Principles

The SQL generator must follow these principles:

1. Generate DDL only.
2. Do not generate DML statements.
3. Use the parameter values defined in this specification.
4. Do not hard-code catalog, schema, or table names when a corresponding parameter exists.
5. Generate a managed Delta table.
6. Generate explicit column definitions.
7. Generate explicit data types.
8. Generate explicit nullability.
9. Generate the system-generated identity column.
10. Generate the primary-key relationship.
11. Generate the defined foreign-key relationships.
12. Generate table and column comments.
13. Use fully qualified object names when referencing parent tables.
14. Use uppercase SQL keywords.
15. Use consistent SQL formatting and indentation.
16. Generate executable Databricks SQL.

---

# 5. Source Table Parameters

This section identifies the source data object associated with the target table.

Although the source table is not used directly to generate the physical DDL structure in every case, it provides the source-data context for the target table and its business columns.

The parameter table contains two columns:

1. **Parameter** — The name of the parameter. Parameter names may be referenced in subsequent sections.
2. **Value** — The value assigned to the parameter.

## 5.1 Source Table Parameters

The following parameters normally remain unchanged unless the source table or its location changes.

| Parameter                  | Value                                        |
| -------------------------- | -------------------------------------------- |
| `SOURCE_TABLE_NAME`        | `dev_catalog.brz_raw_hrs.randhrs1992_2022v1` |
| `SOURCE_TABLE_DESCRIPTION` | `RAND HRS Codebook`                          |
| `SOURCE_CATALOG_NAME`      | `dev_catalog`                                |
| `SOURCE_SCHEMA_NAME`       | `brz_raw_hrs`                                |

### Parameter Usage

* `SOURCE_TABLE_NAME` identifies the complete source table.
* `SOURCE_CATALOG_NAME` identifies the source catalog.
* `SOURCE_SCHEMA_NAME` identifies the source schema.
* `SOURCE_TABLE_DESCRIPTION` provides a human-readable description of the source data.

The SQL generator must not assume that the source and target catalog/schema are necessarily the same.

---

# 6. Target Table Parameters

This section defines the target table and its physical characteristics.

The target-table parameters identify:

* Where the table will be created
* The target table name
* The target table description
* The target table's primary key
* Storage format
* Table type
* Loading pattern

The loading pattern is retained as a table parameter for consistency with the overall CDM specification but is **not used to generate DML in this DDL specification**.

## 6.1 User-Supplied Parameters

The user must provide values for:

* `TARGET_TABLE_NAME`
* `TARGET_TABLE_DESCRIPTION`
* `TARGET_PRIMARY_KEY`

## 6.2 Target Table Parameters

| Parameter                  | Value                                      |
| -------------------------- | ------------------------------------------ |
| `TARGET_TABLE_NAME`        | `fact_demographics`                        |
| `TARGET_TABLE_DESCRIPTION` | `Stores RAND HRS demographic observations` |
| `TARGET_PRIMARY_KEY`       | `fact_demographics_id`                     |
| `TARGET_CATALOG_NAME`      | `dev_catalog`                              |
| `TARGET_SCHEMA_NAME`       | `slv_cdm_hrs`                              |
| `STORAGE_FORMAT`           | `DELTA`                                    |
| `TABLE_TYPE`               | `Managed Table`                            |
| `LOAD_PATTERN`             | `Insert Only`                              |

## 6.3 Parameter Usage

| Parameter                  | Purpose                                                     |
| -------------------------- | ----------------------------------------------------------- |
| `TARGET_TABLE_NAME`        | Identifies the target table name.                           |
| `TARGET_TABLE_DESCRIPTION` | Provides the target table's business description.           |
| `TARGET_PRIMARY_KEY`       | Identifies the target table's system-generated primary key. |
| `TARGET_CATALOG_NAME`      | Identifies the target Databricks catalog.                   |
| `TARGET_SCHEMA_NAME`       | Identifies the target schema.                               |
| `STORAGE_FORMAT`           | Specifies the target storage format.                        |
| `TABLE_TYPE`               | Specifies the target table type.                            |
| `LOAD_PATTERN`             | Documents the expected future DML loading pattern.          |

## 6.4 Fully Qualified Target Table

The fully qualified target table name must be:

`<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.<TARGET_TABLE_NAME>`

For the example parameters:

`dev_catalog.slv_cdm_hrs.fact_demographics`

The SQL generator must use the fully qualified target table name consistently.

---

# 7. Parent Table Dependencies

This section defines the parent tables and relationships required by the target table.

The target table references **system-generated surrogate keys** from the parent tables.

The source natural identifiers are separate from the parent surrogate keys.

## 7.1 Parent Tables

| Object         | Requirement             |
| -------------- | ----------------------- |
| Catalog        | `<TARGET_CATALOG_NAME>` |
| Schema         | `<TARGET_SCHEMA_NAME>`  |
| Parent Table 1 | `hub_respondent`        |
| Parent Table 2 | `dim_wave`              |

The fully qualified parent tables are:

* `<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.hub_respondent`
* `<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.dim_wave`

## 7.2 Parent Table Keys

| Parent Table     | System-Generated Primary Key | Natural Identifier |
| ---------------- | ---------------------------- | ------------------ |
| `hub_respondent` | `respondent_id`              | `HHIDPN`           |
| `dim_wave`       | `wave_id`                    | `wave_number`      |

### Key Architecture

`HHIDPN` is the natural identifier used to identify a respondent.

`respondent_id` is the system-generated surrogate primary key of `hub_respondent`.

`wave_number` is the natural identifier used to identify a survey wave.

`wave_id` is the system-generated surrogate primary key of `dim_wave`.

The target table references the parent surrogate keys, not the source natural identifiers.

## 7.3 Parent-Child Relationships

| Parent Table     | Child Table                                                      | Relationship |
| ---------------- | ---------------------------------------------------------------- | ------------ |
| `hub_respondent` | `<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.<TARGET_TABLE_NAME>` | One-to-Many  |
| `dim_wave`       | `<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.<TARGET_TABLE_NAME>` | One-to-Many  |

From the target-table perspective, each relationship is many-to-one.

## 7.4 Target Foreign Keys

| Target Column   | Parent Table     | Parent Key      |
| --------------- | ---------------- | --------------- |
| `respondent_id` | `hub_respondent` | `respondent_id` |
| `wave_id`       | `dim_wave`       | `wave_id`       |

The generated DDL must establish these relationships using foreign-key constraints.

## 7.5 Natural Identifier Relationships

| Natural Identifier | Parent Table     | Parent Surrogate Key |
| ------------------ | ---------------- | -------------------- |
| `HHIDPN`           | `hub_respondent` | `respondent_id`      |
| `wave_number`      | `dim_wave`       | `wave_id`            |

The mechanism used to resolve these natural identifiers to surrogate keys is outside the scope of this DDL specification.

## 7.6 Business Grain

The target table represents observations at the respondent-by-wave level.

The logical business key is:

`respondent_id + wave_id`

Together, these columns identify the respondent and survey wave represented by the target observation.

Business-key uniqueness validation is defined as a data-validation requirement and is outside the scope of DML generation in this specification.

---

# 8. Target Column Definitions

This section defines all columns that must be included in the target table.

The target table consists of the following column categories:

1. System-generated primary key
2. Parent foreign keys
3. Audit columns
4. Natural identifier columns
5. Business columns

## 8.1 Column Definition Rules

Every target column must define:

* Column name
* Databricks data type
* Nullability
* Description
* Key role, where applicable

## 8.2 Identity Primary Key

| Column                | Type     | Nullable | Generation                     | Description                            |
| --------------------- | -------- | -------- | ------------------------------ | -------------------------------------- |
| `<TARGET_TABLE_NAME>_id` | `BIGINT` | No       | `GENERATED ALWAYS AS IDENTITY` | System-generated surrogate primary key |

The identity column:

* Must use `BIGINT`.
* Must use `GENERATED ALWAYS AS IDENTITY`.
* Must be the target table's primary key.
* Must not receive a value from a future DML source-column mapping.
* Must be generated by Databricks.

## 8.3 Foreign Key Columns

| Column          | Type     | Nullable | Description                                   |
| --------------- | -------- | -------- | --------------------------------------------- |
| `respondent_id` | `BIGINT` | No       | Foreign key to `hub_respondent.respondent_id` |
| `wave_id`       | `BIGINT` | No       | Foreign key to `dim_wave.wave_id`             |

## 8.4 Audit Columns

| Column        | Type      | Nullable | Description                            |
| ------------- | --------- | -------- | -------------------------------------- |
| `create_date` | `DATE`    | No       | Record creation date                   |
| `update_date` | `DATE`    | No       | Date the record was last updated       |
| `active`      | `BOOLEAN` | No       | Indicates whether the record is active |

The DDL specification defines the columns and their nullability only.

The logic used to populate the audit columns is outside the scope of this specification.

## 8.5 Natural Identifier Columns

| Column        | Type     | Nullable | Description                             |
| ------------- | -------- | -------- | --------------------------------------- |
| `HHIDPN`      | `DOUBLE` | No       | RAND HRS natural respondent identifier  |
| `wave_number` | `STRING` | No       | RAND HRS natural survey wave identifier |

`HHIDPN` and `wave_number` are natural identifiers.

They are not surrogate primary keys.

`wave_number` must be defined as `STRING`.

The SQL generator must not convert `wave_number` to an integer type unless explicitly instructed by a future specification.

## 8.6 Business Columns

The following business columns are included in the target table.

| Column     | Databricks Type | Nullable | Description                      |
| ---------- | --------------- | -------- | -------------------------------- |
| `agey_e`   | `DECIMAL(10,2)` | Yes      | Respondent age                   |
| `raracem`  | `TINYINT`       | Yes      | RAND HRS race classification     |
| `rahispan` | `TINYINT`       | Yes      | RAND HRS Hispanic classification |
| `cenreg`   | `TINYINT`       | Yes      | Census region                    |
| `raedyrs`  | `TINYINT`       | Yes      | Years of education               |
| `mstat`    | `TINYINT`       | Yes      | Marital status                   |
| `rarelig`  | `TINYINT`       | Yes      | Religion                         |
| `ravetrn`  | `TINYINT`       | Yes      | Veteran status                   |

The source variables and transformation rules used to populate these columns are not part of the DDL specification.

They will be defined in the future DML specification.

---

# 9. Source-to-Target Metadata

This section documents the relationship between RAND source variables and target columns for purposes of data-model documentation.

It does **not** define DML transformation logic.

## 9.1 Wave-Specific Variables

Wave-specific RAND variables contain a wave identifier in the source variable name.

Examples include:

* `R1AGEY_E`
* `R2AGEY_E`
* `R3AGEY_E`
* ...
* `R16AGEY_E`

These variables correspond to the target business column:

`agey_e`

The target column itself occurs once in the target table.

## 9.2 Wave-Invariant Variables

Wave-invariant variables occur once per respondent in the source data and do not contain an `R{n}` wave prefix.

Examples include:

* `RARACEM`
* `RAHISPAN`
* `RAEDYRS`
* `RARELIG`
* `RAVETRN`

These variables correspond to the target business columns:

* `raracem`
* `rahispan`
* `raedyrs`
* `rarelig`
* `ravetrn`

The method used to associate these variables with target observations is outside the scope of this DDL specification.

## 9.3 Source Mapping Metadata

| Target Column | Source Variable Pattern | RAND Type | Databricks Type |
| ------------- | ----------------------- | --------- | --------------- |
| `agey_e`      | `R{n}AGEY_E`            | `CONT`    | `DECIMAL(10,2)` |
| `raracem`     | `RARACEM`               | `CATEG`   | `TINYINT`       |
| `rahispan`    | `RAHISPAN`              | `CATEG`   | `TINYINT`       |
| `cenreg`      | `R{n}CENREG`            | `CATEG`   | `TINYINT`       |
| `raedyrs`     | `RAEDYRS`               | `CATEG`   | `TINYINT`       |
| `mstat`       | `R{n}MSTAT`             | `CATEG`   | `TINYINT`       |
| `rarelig`     | `RARELIG`               | `CATEG`   | `TINYINT`       |
| `ravetrn`     | `RAVETRN`               | `CATEG`   | `TINYINT`       |

`{n}` represents the applicable survey wave.

---

# 10. Constraint Requirements

This section defines the logical constraints that must be represented in the target-table DDL.

## 10.1 Constraint Categories

| Constraint Type | Required             | Purpose                                               |
| --------------- | -------------------- | ----------------------------------------------------- |
| `PRIMARY KEY`   | Yes                  | Identifies the target table's primary key             |
| `FOREIGN KEY`   | Yes                  | Documents parent-child relationships                  |
| `UNIQUE`        | No                   | Business-key documentation only if supported/required |
| `NOT NULL`      | Yes, where specified | Prevents NULL values                                  |
| `CHECK`         | Optional             | Defines explicit data-quality rules                   |
| `IDENTITY`      | Yes                  | Generates surrogate key values                        |

### Databricks Constraint Behavior

The SQL generator must distinguish between **enforced constraints** and **informational constraints**.

`NOT NULL` and applicable `CHECK` constraints are enforced by Databricks.

Primary-key and foreign-key constraints are informational and document the logical data model; they should not be described in generated documentation as enforcing referential integrity or uniqueness.

The generated specification must not claim that primary-key, foreign-key, or business-key constraints provide enforcement unless the target Databricks environment explicitly supports such enforcement.

## 10.2 Primary Key

The target table must define:

`<TARGET_PRIMARY_KEY>`

as its primary key.

Recommended constraint name:

`pk_<TARGET_TABLE_NAME>`

## 10.3 Foreign Keys

The target table must define:

`respondent_id`

as a foreign key to:

`<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.hub_respondent(respondent_id)`

The target table must define:

`wave_id`

as a foreign key to:

`<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.dim_wave(wave_id)`

Recommended constraint names:

* `fk_<TARGET_TABLE_NAME>_hub_respondent`
* `fk_<TARGET_TABLE_NAME>_dim_wave`

## 10.4 Business Key

The logical business key is:

`(respondent_id, wave_id)`

Recommended constraint name, if a supported UNIQUE constraint is explicitly required:

`uq_<TARGET_TABLE_NAME>_respondent_wave`

However, the DDL generator must not assume that this constraint provides enforcement in the target Databricks Runtime.

Business-key validation belongs to the future DML/data-quality specification.

---

# 11. Column Constraint Rules

## 11.1 Identity Column

The target primary-key column must be defined as:

```sql
<TARGET_PRIMARY_KEY> BIGINT
    GENERATED ALWAYS AS IDENTITY
```

The identity column must not be populated by future DML.

## 11.2 Foreign Key Columns

`respondent_id`:

```sql
respondent_id BIGINT NOT NULL
```

`wave_id`:

```sql
wave_id BIGINT NOT NULL
```

The corresponding foreign-key constraints must reference the parent surrogate keys.

## 11.3 Audit Columns

The following columns must be `NOT NULL`:

```text
create_date
update_date
active
```

## 11.4 Natural Identifier Columns

The following columns must be `NOT NULL`:

```text
HHIDPN
wave_number
```

## 11.5 Business Columns

Business columns must use the nullability specified in Section 8.

Unless explicitly specified otherwise, RAND HRS business attributes should remain nullable because a source value may not exist for a particular respondent or observation.

---

# 12. Table and Column Comments

The generated DDL must contain a comment for the target table.

## 12.1 Table Comment

The table comment must be derived from:

`TARGET_TABLE_DESCRIPTION`

Example:

```sql
COMMENT 'Stores RAND HRS demographic observations'
```

## 12.2 Column Comments

The generated DDL must include comments for all target columns.

Column comments should be based on the descriptions provided in this specification.

Example:

```text
fact_demographics_id
→ System-generated surrogate primary key

respondent_id
→ Foreign key to hub_respondent.respondent_id

wave_id
→ Foreign key to dim_wave.wave_id

HHIDPN
→ RAND HRS natural respondent identifier

wave_number
→ RAND HRS natural survey wave identifier
```

---

# 13. DDL Generation Requirements

The generated DDL must satisfy the following requirements.

| Requirement                    | Required |
| ------------------------------ | -------- |
| `DROP TABLE IF EXISTS`         | Yes      |
| `CREATE TABLE`                 | Yes      |
| `USING DELTA`                  | Yes      |
| Managed Table                  | Yes      |
| Identity Column                | Yes      |
| `GENERATED ALWAYS AS IDENTITY` | Yes      |
| Primary Key                    | Yes      |
| Foreign Keys                   | Yes      |
| Explicit Column Definitions    | Yes      |
| Explicit Data Types            | Yes      |
| Explicit Nullability           | Yes      |
| Table Comment                  | Yes      |
| Column Comments                | Yes      |
| Fully Qualified Parent Tables  | Yes      |
| Uppercase SQL Keywords         | Yes      |
| Consistent Indentation         | Yes      |
| Semicolon Termination          | Yes      |

## 13.1 Statement Order

The generated DDL must use the following logical order:

1. Drop existing target table.
2. Create target table.
3. Define all columns.
4. Define the identity primary key.
5. Define foreign keys.
6. Define other applicable constraints.
7. Define table and column comments where required.

## 13.2 DROP TABLE Requirement

The generated SQL must begin with:

```sql
DROP TABLE IF EXISTS <TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.<TARGET_TABLE_NAME>;
```

This requirement ensures that the DDL can recreate the target table during development and deployment.

## 13.3 CREATE TABLE Requirement

The generated SQL must create the target as a managed Delta table.

The generated statement must not specify an external storage location unless explicitly required by a future version of this specification.

---

# 14. Unsupported DDL Objects

The DDL generator must not generate the following unless explicitly requested by a future specification:

* Views
* Materialized views
* Indexes
* External tables
* Table partitions
* Z-ORDER specifications
* `OPTIMIZE`
* Stored procedures
* Functions
* Triggers

The DDL generator must not generate DML statements, including:

* `INSERT`
* `UPDATE`
* `DELETE`
* `MERGE`

---

# 15. SQL Formatting Requirements

The generated SQL must conform to the following formatting rules.

## 15.1 Keywords

SQL keywords must be uppercase.

Examples:

```sql
CREATE TABLE
DROP TABLE
PRIMARY KEY
FOREIGN KEY
REFERENCES
GENERATED ALWAYS AS IDENTITY
USING DELTA
```

## 15.2 Indentation

Use consistent four-space indentation.

## 15.3 Column Definitions

Each column must appear on its own line.

## 15.4 Constraint Definitions

Each table-level constraint must appear on its own logical block.

## 15.5 Object Qualification

Parent tables must be fully qualified using:

`<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.<TABLE_NAME>`

## 15.6 SQL Termination

Every SQL statement must terminate with a semicolon.

---

# 16. DDL Validation Requirements

Validation must confirm that the generated DDL creates the target table with the required structure.

## 16.1 Table Validation

| Validation          | Required |
| ------------------- | -------- |
| Target table exists | Yes      |
| Correct catalog     | Yes      |
| Correct schema      | Yes      |
| Correct table name  | Yes      |
| Delta format        | Yes      |
| Managed table       | Yes      |

## 16.2 Column Validation

| Validation                           | Required |
| ------------------------------------ | -------- |
| Target primary-key column exists     | Yes      |
| Identity column exists               | Yes      |
| `respondent_id` exists               | Yes      |
| `wave_id` exists                     | Yes      |
| `create_date` exists                 | Yes      |
| `update_date` exists                 | Yes      |
| `active` exists                      | Yes      |
| `HHIDPN` exists                      | Yes      |
| `wave_number` exists                 | Yes      |
| All specified business columns exist | Yes      |
| Data types are correct               | Yes      |
| Nullability is correct               | Yes      |

## 16.3 Constraint Validation

| Validation                                | Required |
| ----------------------------------------- | -------- |
| Primary key definition exists             | Yes      |
| Respondent foreign key exists             | Yes      |
| Wave foreign key exists                   | Yes      |
| Constraint names follow naming convention | Yes      |

## 16.4 Parent Table Validation

The following parent objects must exist before the target DDL is successfully deployed:

| Object                         | Required |
| ------------------------------ | -------- |
| `hub_respondent`               | Yes      |
| `dim_wave`                     | Yes      |
| `hub_respondent.respondent_id` | Yes      |
| `dim_wave.wave_id`             | Yes      |

## 16.5 Validation Scope

This section validates the **DDL and resulting table structure only**.

The following validations are explicitly excluded:

* Duplicate business keys
* Source-to-target row counts
* Referential integrity of loaded data
* NULL values in loaded data
* Transformation accuracy
* Source-to-target value comparisons

These validations belong to the future DML/data-quality specification.

---

# 17. DDL Deliverables

The SQL generation process must produce the following deliverable.

| Item           | Required Value                            |
| -------------- | ----------------------------------------- |
| SQL File       | `/sql/ddl/create_<TARGET_TABLE_NAME>.sql` |
| Output Format  | SQL only                                  |
| SQL Dialect    | Databricks SQL / Spark SQL                |
| Storage Format | Delta                                     |
| Table Type     | Managed Table                             |
| Target Catalog | `<TARGET_CATALOG_NAME>`                   |
| Target Schema  | `<TARGET_SCHEMA_NAME>`                    |

The generated SQL file must contain all DDL required to create the target table.

The output must not contain explanatory prose outside SQL comments.

---

# 18. Generated DDL Structure

The generated DDL should follow this logical structure:

```sql
DROP TABLE IF EXISTS <TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.<TARGET_TABLE_NAME>;

CREATE TABLE <TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.<TARGET_TABLE_NAME>
(
    <TARGET_PRIMARY_KEY> BIGINT
        GENERATED ALWAYS AS IDENTITY,

    respondent_id BIGINT NOT NULL,

    wave_id BIGINT NOT NULL,

    create_date DATE NOT NULL,

    update_date DATE NOT NULL,

    active BOOLEAN NOT NULL,

    HHIDPN DOUBLE NOT NULL,

    wave_number STRING NOT NULL,

    <BUSINESS_COLUMN_DEFINITIONS>,

    CONSTRAINT pk_<TARGET_TABLE_NAME>
        PRIMARY KEY (<TARGET_PRIMARY_KEY>),

    CONSTRAINT fk_<TARGET_TABLE_NAME>_respondent
        FOREIGN KEY (respondent_id)
        REFERENCES <TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.hub_respondent (respondent_id),

    CONSTRAINT fk_<TARGET_TABLE_NAME>_wave
        FOREIGN KEY (wave_id)
        REFERENCES <TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.dim_wave (wave_id)
)
USING DELTA
COMMENT '<TARGET_TABLE_DESCRIPTION>';
```

The exact generated SQL syntax must be compatible with the specified Databricks Runtime.

---

# 19. Generation Rules Summary

The SQL generator must apply the following rules.

### Rule 1 — DDL Only

Generate DDL only. Do not generate DML.

### Rule 2 — Target Table

Create:

`<TARGET_CATALOG_NAME>.<TARGET_SCHEMA_NAME>.<TARGET_TABLE_NAME>`

as a managed Delta table.

### Rule 3 — Surrogate Primary Key

Use:

`<TARGET_PRIMARY_KEY> BIGINT GENERATED ALWAYS AS IDENTITY`

as the target table's system-generated surrogate primary key.

### Rule 4 — Respondent Relationship

Use:

`respondent_id`

as the foreign key referencing:

`hub_respondent.respondent_id`

### Rule 5 — Wave Relationship

Use:

`wave_id`

as the foreign key referencing:

`dim_wave.wave_id`

### Rule 6 — Natural Identifiers

Define:

`HHIDPN` as `DOUBLE`

and:

`wave_number` as `STRING`.

### Rule 7 — Business Grain

The logical business grain is:

`respondent_id + wave_id`

### Rule 8 — Audit Columns

Include:

* `create_date`
* `update_date`
* `active`

### Rule 9 — Business Columns

Generate all business columns defined in the target-column specification.

### Rule 10 — Comments

Generate table and column comments.

### Rule 11 — Constraints

Generate required primary-key and foreign-key definitions.

### Rule 12 — Validation

Validate the resulting table structure and DDL-defined constraints.

### Rule 13 — DML Separation

Do not generate source-to-target transformation or loading logic.

### Rule 14 — Future DML

DML requirements will be defined separately and may reference the table structure, keys, natural identifiers, and business columns defined by this specification.

---

# 20. Future DML Specification Boundary

The future DML specification should reference this DDL specification rather than redefine the target-table structure.

The future DML specification should define:

* Source-to-target mappings
* Parent-table lookup logic
* `HHIDPN` → `respondent_id` resolution
* `wave_number` → `wave_id` resolution
* Wave-specific transformations
* Wave-invariant transformations
* Unpivot logic
* NULL handling
* Audit-column population
* Insert logic
* Duplicate handling
* Business-key validation
* Data-quality rules
* Row-count validation
* Source-to-target reconciliation

The DML specification must use the target table structure established by this DDL specification.

---

# 21. Final Generation Instruction

Generate a complete Databricks SQL DDL script from this specification.

The generated script must:

1. Create the specified managed Delta table.
2. Use the specified catalog, schema, and table name.
3. Create the system-generated identity primary key.
4. Define all required columns.
5. Apply the specified data types and nullability.
6. Define the respondent foreign key.
7. Define the wave foreign key.
8. Define applicable constraints.
9. Include table and column comments.
10. Follow the specified SQL formatting requirements.
11. Use only DDL statements.
12. Not generate INSERT, UPDATE, DELETE, MERGE, or other DML.
13. Not invent columns, constraints, transformations, or business rules that are not defined in this specification.
14. Produce the final SQL as the sole deliverable.

**Output:** SQL only.
